# End-to-End Tiered Data Management Pipeline (Google Colab Runner)

Reproducing a modular prototype of **Tiered Data Management** for LLM pre-training data based on:
> **"Data Science and Technology Towards AGI Part I: Tiered Data Management"**  
> *(arXiv:2602.09003)* — [https://arxiv.org/abs/2602.09003](https://arxiv.org/abs/2602.09003)

### Pipeline Overview:
- **Phase 1 (L1: Clean):** Heuristic text cleaning, length/ratio filtering & exact SHA-256 deduplication.
- **Phase 2 (L2: Selected):** Weak demo labeling, TF-IDF + numeric feature selector, scoring & selection.
- **Phase 3 (L3: Refined):** Deterministic offline MockLLM synthesis (cleaned text, Q&A, textbook chapters).
- **Phase 3.5 (L4: Organized):** Knowledge unit validation & structured export with provenance metadata.
- **Phase 4:** Cross-tier evaluation, retention analysis & multi-format report generation.

---  
## Step 1: Environment Setup & Repository Clone
Clone the official repository or pull the latest commits if already present.

In [25]:
# Cell 1: Clone repository (recommended) or pull latest updates
import os

repo_url = "https://github.com/mahendravelagapudi099-wq/Ultra-Data.git"
target_dir = "/content/Ultra-Data"

if not os.path.exists(target_dir):
    !git clone {repo_url} {target_dir}
else:
    print(f"{target_dir} already exists. Pulling latest updates...")
    !cd {target_dir} && git pull origin main

# Optional: Mount Google Drive if syncing via Drive
# from google.colab import drive
# drive.mount('/content/drive')

/content/Ultra-Data already exists. Pulling latest updates...
From https://github.com/mahendravelagapudi099-wq/Ultra-Data
 * branch            main       -> FETCH_HEAD
Already up to date.


---  
## Step 2: Navigate to Project Directory
Set the current working directory to the repository root.

In [26]:
# Cell 2: Change directory into the project root
import os

candidate_paths = [
    "/content/Ultra-Data",
    "/content/drive/MyDrive/Ultra-Data",
    "/content/drive/MyDrive/Ultra-Dataa",
]

for path in candidate_paths:
    if os.path.exists(path):
        os.chdir(path)
        print(f"Active project root: {path}")
        break
else:
    print(f"Current working directory: {os.getcwd()}")

!pwd
!ls -la

Active project root: /content/Ultra-Data
/content/Ultra-Data
total 96
drwxr-xr-x 10 root root  4096 Sep 14 11:18 .
drwxr-xr-x  1 root root  4096 Sep 14 11:18 ..
-rw-r--r--  1 root root  6494 Sep 14 11:18 AGENTS.md
drwxr-xr-x  2 root root  4096 Sep 14 11:18 colab
drwxr-xr-x  2 root root  4096 Sep 14 11:18 configs
drwxr-xr-x  8 root root  4096 Sep 14 11:18 data
drwxr-xr-x  8 root root  4096 Sep 14 11:22 .git
-rw-r--r--  1 root root   542 Sep 14 11:18 .gitignore
drwxr-xr-x  2 root root  4096 Sep 14 11:18 paper
-rw-r--r--  1 root root  7600 Sep 14 11:18 PHASES.md
-rw-r--r--  1 root root 16129 Sep 14 11:18 README.md
drwxr-xr-x  2 root root  4096 Sep 14 11:18 reports
-rw-r--r--  1 root root  2690 Sep 14 11:18 requirements.lock.txt
-rw-r--r--  1 root root   733 Sep 14 11:18 requirements-ml.txt
-rw-r--r--  1 root root   212 Sep 14 11:18 requirements.txt
-rw-r--r--  1 root root  7175 Sep 14 11:18 RESULTS_TEMPLATE.md
drwxr-xr-x  2 root root  4096 Sep 14 11:18 scripts
drwxr-xr-x  6 root root  409

---  
## Step 3: Install Required Dependencies
Installs Typer, Rich, Scikit-learn, Hugging Face Datasets, Pandas, and PyYAML.

In [27]:
# Cell 3: Install required packages if missing
!pip install -q -r requirements.txt

---  
## Step 4: Configure Python Search Path (PYTHONPATH)
Ensures Python locates the `src` package and utility modules.

In [28]:
# Cell 4: Set PYTHONPATH to project root
import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.environ["PYTHONPATH"] = "."
print(f"Project root (CWD): {project_root}")
print(f"PYTHONPATH: {os.environ.get('PYTHONPATH')}")

Project root (CWD): /content/Ultra-Data
PYTHONPATH: .


---  
## Step 5: (Optional) Ingest Real Web Sample from Hugging Face
Streams real web documents from `openbmb/Ultra-FineWeb` (150 records) directly to `data/l0_raw/l0_real_sample.jsonl`.  
*Note: If offline, rate-limited, or interrupted, the next step automatically falls back to local substitute data.*

In [29]:
# Cell 4.5: Fetch real web sample from openbmb/Ultra-FineWeb
# Writes records incrementally to disk with immediate flush
!PYTHONPATH=. python scripts/load_real_data.py -n 150

────────────────── Fetching Real Web Sample from Hugging Face ──────────────────
Dataset: openbmb/Ultra-FineWeb (split: train)
Target count: 150 records (streaming mode)
Destination: /content/Ultra-Data/data/l0_raw/l0_real_sample.jsonl

Connecting to openbmb/Ultra-FineWeb (split: train)...
Resolving data files: 100% 2048/2048 [00:00<00:00, 9305.14it/s]
Resolving data files: 100% 256/256 [00:00<00:00, 6422.55it/s]
Resolving data files: 100% 2048/2048 [00:00<00:00, 9587.27it/s]
Resolving data files: 100% 256/256 [00:00<00:00, 7298.76it/s]
Split 'train' unavailable (Bad split: train. Available splits: ['en', 'zh']). 
Trying fallback...
Ultra-FineWeb unavailable. Trying HuggingFaceFW/fineweb (sample-10BT)...
Resolving data files: 100% 27468/27468 [00:02<00:00, 13357.33it/s]
Resolving data files: 100% 27468/27468 [00:02<00:00, 9340.01it/s] 
['train']
The pipeline will gracefully fall back to local substitute data.
Graceful exit — ready for fallback pipeline execution.


In [30]:
import re
from pathlib import Path

script_path = Path('scripts/load_real_data.py')
content = script_path.read_text()

# Fix the AttributeError: replace args.count with the likely intended args.n based on typical parser conventions or the script's own -n flag
fixed_content = content.replace('args.count', 'args.n')

script_path.write_text(fixed_content)
print("Applied fix to scripts/load_real_data.py: replaced args.count with args.n")

Applied fix to scripts/load_real_data.py: replaced args.count with args.n


### Retry Step 5: Ingest Real Web Sample
Now that the script is patched, we can attempt to fetch the 150 records from Hugging Face again.

In [31]:
!PYTHONPATH=. python scripts/load_real_data.py -n 150

────────────────── Fetching Real Web Sample from Hugging Face ──────────────────
Dataset: openbmb/Ultra-FineWeb (split: train)
Target count: 150 records (streaming mode)
Destination: /content/Ultra-Data/data/l0_raw/l0_real_sample.jsonl

Connecting to openbmb/Ultra-FineWeb (split: train)...
Resolving data files: 100% 2048/2048 [00:00<00:00, 22712.62it/s]
Resolving data files: 100% 256/256 [00:00<00:00, 16605.71it/s]
Resolving data files: 100% 2048/2048 [00:00<00:00, 22583.10it/s]
Resolving data files: 100% 256/256 [00:00<00:00, 22780.14it/s]
Split 'train' unavailable (Bad split: train. Available splits: ['en', 'zh']). 
Trying fallback...
Ultra-FineWeb unavailable. Trying HuggingFaceFW/fineweb (sample-10BT)...
Resolving data files: 100% 27468/27468 [00:01<00:00, 21704.44it/s]
Resolving data files: 100% 27468/27468 [00:01<00:00, 20051.30it/s]
['train']
The pipeline will gracefully fall back to local substitute data.
Graceful exit — ready for fallback pipeline execution.


In [40]:
import re
from pathlib import Path

script_path = Path('scripts/load_real_data.py')
content = script_path.read_text()

# Fix the attribute name error
content = content.replace('args.count', 'args.n')

# Use regex to replace split='train' or split="train" with 'en'
# This covers function arguments and variable assignments even with whitespace
content = re.sub(r"split\s*=\s*['\"]train['\"]", "split='en'", content)

script_path.write_text(content)
print("Successfully forced split='en' in scripts/load_real_data.py using regex.")

Fixed scripts/load_real_data.py: Updated arg attribute and set split to 'en'.


### Rerunning Ingestion and Full Pipeline
Now that we've updated the split name, let's ingest the real data and update the reports.

In [41]:
!PYTHONPATH=. python scripts/load_real_data.py -n 150
!PYTHONPATH=. python scripts/run_phase1.py
!PYTHONPATH=. python scripts/run_l2.py --config configs/l2_tiny.yaml
!PYTHONPATH=. python scripts/run_l3.py --config configs/l3_tiny.yaml
!PYTHONPATH=. python scripts/run_l4_export.py --config configs/l4_tiny.yaml
!PYTHONPATH=. python scripts/run_evaluation.py

────────────────── Fetching Real Web Sample from Hugging Face ──────────────────
Dataset: openbmb/Ultra-FineWeb (split: train)
Target count: 150 records (streaming mode)
Destination: /content/Ultra-Data/data/l0_raw/l0_real_sample.jsonl

Connecting to openbmb/Ultra-FineWeb (split: train)...
Resolving data files: 100% 2048/2048 [00:00<00:00, 19093.45it/s]
Resolving data files: 100% 256/256 [00:00<00:00, 21947.14it/s]
Resolving data files: 100% 2048/2048 [00:00<00:00, 22312.79it/s]
Resolving data files: 100% 256/256 [00:00<00:00, 21798.77it/s]
Split 'train' unavailable (Bad split: train. Available splits: ['en', 'zh']). 
Trying fallback...
Ultra-FineWeb unavailable. Trying HuggingFaceFW/fineweb (sample-10BT)...
Resolving data files: 100% 27468/27468 [00:01<00:00, 14871.80it/s]
Resolving data files: 100% 27468/27468 [00:01<00:00, 20128.62it/s]
['train']
The pipeline will gracefully fall back to local substitute data.
Graceful exit — ready for fallback pipeline execution.
[Phase 1] No real 

In [34]:
from pathlib import Path
report = Path("reports/pipeline_summary.md")
if report.exists():
    print(report.read_text())

# Pipeline Evaluation & Tier Comparison Report

> [!NOTE]
> **Notice:** This report reflects a small-scale, offline demo reproduction of the methodology from
> *"Data Science and Technology Towards AGI Part I: Tiered Data Management"* (arXiv:2602.09003).
> Heuristics, thresholds, and mock LLM synthesizers are lightweight starter implementations and do not claim paper-scale results.

**Generated:** 2026-09-14 11:23:47 UTC  
**Data Source:** `Local expanded substitute dataset (deterministic offline)`  
**Environment:** Colab / Linux / Local agnostic  

## 1. Tier-by-Tier Quality Progression

| Tier | Description | Doc Count | Avg Chars | Avg Words | Alpha Ratio | Symbol Ratio | Duplicates |
|---|---|---|---|---|---|---|---|
| `L0_Raw` | Unfiltered raw web corpus (real sample or substitute) | 145 | 181.9 | 24.0 | 0.7996 | 0.0762 | 10 |
| `L1_Filtered` | Heuristic filtered & exact deduped (clean text) | 33 | 282.8 | 38.2 | 0.8428 | 0.0248 | 0 |
| `L2_Selected` | Model-selected informative 

---  
## Step 6: Phase 1 — Smart Ingestion & L1 Heuristic Filtering
Uses `scripts/run_phase1.py` as the smart orchestrator:  
- If `l0_real_sample.jsonl` exists from Step 5, ingests and filters the real web data.  
- If absent, automatically generates a synthetic local substitute dataset and filters it.  
- Applies NFKC normalization, boilerplate stripping, heuristic threshold gates, and SHA-256 deduplication.

In [35]:
# Cell 5: Run Phase 1 via smart orchestrator (run_phase1.py)
!PYTHONPATH=. python scripts/run_phase1.py

[Phase 1] No real data found — generating local substitute data...
Generated 145 raw substitute records at /content/Ultra-Data/data/l0_raw/l0_expanded_SUBSTITUTE.jsonl
Source: local_substitute_l1
[Phase 1] Running L1 filtering with config: /content/Ultra-Data/configs/l1_expanded.yaml
───────────────────── L1 Tiny Demo — Tier 1 Data Cleaning ──────────────────────
Config: /content/Ultra-Data/configs/l1_expanded.yaml
[L1] Warning: Real data file 'data/l0_raw/l0_real_sample.jsonl' not found or empty.
[L1] Automatically falling back to local substitute data...
[L1] Using local substitute data: /content/Ultra-Data/data/l0_raw/l0_expanded_SUBSTITUTE.jsonl
L1 processing: 100% 145/145 [00:00<00:00, 11002.30doc/s]
         L1 Pipeline Statistics          
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Metric             ┃            Value ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Input Count        │              145 │
│ Output Count       │               33 │
│ Removed (filtered) │           

---  
## Step 7: Phase 2 — L2 Model-Driven Selection
Trains a lightweight selector using TF-IDF n-grams and scaled text statistics (fitted strictly on the training split to avoid data leakage) to score and prioritize high-value educational tokens.

In [36]:
# Cell 6: Run Phase 2 (L2 Model-Driven Selection)
!PYTHONPATH=. python scripts/run_l2.py --config configs/l2_tiny.yaml

─────────────────── L2 Demo — Tier 2 Model-Driven Selection ────────────────────
Config: configs/l2_tiny.yaml
         L2 Selection Statistics          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric                        ┃  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Input Records (L1)            │     33 │
│ Selected Records (L2)         │     16 │
│ Selection Rate                │  48.5% │
│ Selector Train Accuracy       │  95.7% │
│ Selector Test Accuracy        │ 100.0% │
│ Mean Quality Score (All)      │ 0.3570 │
│ Mean Quality Score (Selected) │ 0.6605 │
└───────────────────────────────┴────────┘

Output Files:
  All Scored (Parquet): 
/content/Ultra-Data/data/l2_scores/l2_selected_all_scored.parquet
  Selected (Parquet):   /content/Ultra-Data/data/l2_selected/l2_selected.parquet
  Selected (JSONL):     /content/Ultra-Data/data/l2_selected/l2_selected.jsonl
───────────────────────────────────── Done ─────────────────────────────────────


---  
## Step 8: Phase 3 & 3.5 — L3 Refinement & L4 Knowledge Export
- **Phase 3:** Uses deterministic offline `MockLLMProvider` to transform raw selected text into structured educational assets (overview article, conceptual Q&A pair, textbook chapter).  
- **Phase 3.5:** Applies structural quality gates and exports standardized knowledge units with UTC provenance metadata.

In [37]:
# Cell 7: Run Phase 3 (Refinement) & Phase 3.5 (Organized Knowledge Export)
!PYTHONPATH=. python scripts/run_l3.py --config configs/l3_tiny.yaml
!PYTHONPATH=. python scripts/run_l4_export.py --config configs/l4_tiny.yaml

───────────────── L3 Demo — Tier 3 LLM Refinement & Synthesis ──────────────────
Config: configs/l3_tiny.yaml
L3 Refinement: 100% 16/16 [00:00<00:00, 5594.27doc/s]
                          L3 Refinement Statistics                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                      ┃                                      Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Input Records (L2 Selected) │                                         16 │
│ Refined Records (L3)        │                                         16 │
│ Synthesis Generator         │ MockLLMProvider (mock-rule-synthesizer-v1) │
└─────────────────────────────┴────────────────────────────────────────────┘

Output Files:
  Parquet: /content/Ultra-Data/data/l3_refined/l3_refined.parquet
  JSONL:   /content/Ultra-Data/data/l3_refined/l3_refined.jsonl
───────────────────────────────────── Done ────────────────────────────────

---  
## Step 9: Phase 4 — Cross-Tier Evaluation & Reporting
Computes tier-by-tier metrics (document counts, word counts, alphabetic/symbol ratios, duplicate counts, and transition rates) and saves Markdown, CSV, and JSON reports to `reports/`.

In [38]:
# Cell 8: Run Phase 4 Evaluation across all tiers
!PYTHONPATH=. python scripts/run_evaluation.py

───────────────── Pipeline Evaluation — Cross-Tier Progression ─────────────────
Data Source: Local expanded substitute dataset (deterministic offline)

                         Cross-Tier Progression Summary                         
┏━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃             ┃           ┃           ┃             ┃      Symbol ┃            ┃
┃ Tier        ┃ Doc Count ┃ Avg Words ┃ Alpha Ratio ┃       Ratio ┃ Duplicates ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ L0_Raw      │       145 │      24.0 │      0.7996 │      0.0762 │         10 │
│ L1_Filtered │        33 │      38.2 │      0.8428 │      0.0248 │          0 │
│ L2_Selected │        16 │      43.8 │      0.8377 │      0.0251 │          0 │
│ L3_Refined  │        16 │      43.8 │      0.8377 │      0.0251 │          0 │
│ L4_Organiz… │        16 │      49.4 │      0.8251 │      0.0349 │          0 │
└─────────────┴───────────┴──────────

---  
## Step 10: Inspect Evaluation Report
Displays the final generated Markdown report summarizing quality progression across tiers.

In [39]:
# Cell 9: Display the generated cross-tier summary report
from pathlib import Path

report_path = Path("reports/pipeline_summary.md")
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
else:
    print("Report not found at reports/pipeline_summary.md. Ensure Phase 4 ran successfully.")

# Pipeline Evaluation & Tier Comparison Report

> [!NOTE]
> **Notice:** This report reflects a small-scale, offline demo reproduction of the methodology from
> *"Data Science and Technology Towards AGI Part I: Tiered Data Management"* (arXiv:2602.09003).
> Heuristics, thresholds, and mock LLM synthesizers are lightweight starter implementations and do not claim paper-scale results.

**Generated:** 2026-09-14 11:24:00 UTC  
**Data Source:** `Local expanded substitute dataset (deterministic offline)`  
**Environment:** Colab / Linux / Local agnostic  

## 1. Tier-by-Tier Quality Progression

| Tier | Description | Doc Count | Avg Chars | Avg Words | Alpha Ratio | Symbol Ratio | Duplicates |
|---|---|---|---|---|---|---|---|
| `L0_Raw` | Unfiltered raw web corpus (real sample or substitute) | 145 | 181.9 | 24.0 | 0.7996 | 0.0762 | 10 |
| `L1_Filtered` | Heuristic filtered & exact deduped (clean text) | 33 | 282.8 | 38.2 | 0.8428 | 0.0248 | 0 |
| `L2_Selected` | Model-selected informative 

---  
## Next Steps & References

- **Architecture & Methodology:** See [PHASES.md](PHASES.md) for a detailed mapping of pipeline components to the research paper.
- **Schema Specifications:** See [RESULTS_TEMPLATE.md](RESULTS_TEMPLATE.md) for exact column definitions across all tiers.
- **Paper Reference:** *"Data Science and Technology Towards AGI Part I: Tiered Data Management"* ([arXiv:2602.09003](https://arxiv.org/abs/2602.09003)).